# LAB | Micromanage your agents

**Scenario:** you are building an evaluation system for **one specific use case**. Public benchmarks (like HLE in class) don't reflect your client's real questions, so you build **your own dataset**, upload it to LangSmith, run experiments and report what you found.

**How to use this notebook:**
- Cells marked **`# TODO`** are yours to complete. Replace every `...` with your code.
- **Hints** point to the lesson notebook (`02_langsmith(1).ipynb`), where you saw each idea.
- Markdown cells with a bold prompt (e.g. **Document your choice here:**, **Key findings:**) are for your written answers.
- Run the cells **in order**, top to bottom.

| Part | You will… | Concept from class |
|---|---|---|
| 1 | Choose a domain + set up LangSmith | Tracing, projects, regions |
| 2 | Build and upload your own dataset | Datasets: inputs, reference outputs, metadata |
| 3 | Write the target function + evaluators | Target, LLM-as-a-judge (OpenEvals), exact match |
| 4 | Run the experiment and review it in the UI | `client.evaluate()`, experiments, feedback |
| 5 | Analyse results + write `evaluation_summary.md` | Metrics, error analysis |
| Optional 1 | Design a custom evaluator | Judge prompt anatomy (notebook 01) |
| Optional 2 | A/B test models on cost vs performance | Model comparison, cost per correct answer |

In [ ]:
%pip install langsmith openai openevals python-dotenv pandas matplotlib -q

## Part 1: Setup & document selection (30–45 min)
### Step 1: Choose your domain and documents

Pick a domain where **you** can judge what a good answer looks like. Ideas: customer support FAQ, a company policy, a product catalogue, course material, a legal or compliance text, code review…

Each example needs:
- a clear **input** (question, ticket, task)
- ideally an **expected output** (reference answer, label, summary)
- enough **variety**: easy and hard cases, different categories

 **Hint:** the best datasets include **tricky cases**: questions with no answer in the document, ambiguous wording, multi-step questions. Easy-only datasets give 100% scores and teach you nothing (remember the calibration set in notebook 01, §6).

** Document your choice here:**
- Domain (2–3 sentences):
- Source(s) of my documents (name + URL or file):
- What is one example? (input → expected output):
- What makes a *good* answer in my domain?

### Step 2: Set up LangSmith

 **Hint:** copy the setup pattern from `02_langsmith(1).ipynb` (Setup: keys and region). Your `.env` needs `LANGSMITH_API_KEY` and `OPENAI_API_KEY`. Check your **region**: `eu.smith.langchain.com` in the browser means EU.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# TODO: turn tracing on, choose a descriptive project name, set your region
os.environ["LANGSMITH_TRACING"] = ...
PROJECT_NAME = ...                      # e.g. "lab-langsmith-<domain>"
os.environ["LANGSMITH_PROJECT"] = PROJECT_NAME
os.environ["LANGSMITH_ENDPOINT"] = ...  # US or EU endpoint

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from langsmith import Client, traceable
from langsmith.wrappers import wrap_openai
from openai import OpenAI
from openevals.llm import create_llm_as_judge
from openevals.prompts import CORRECTNESS_PROMPT

OUT = Path("outputs"); OUT.mkdir(exist_ok=True)   # charts and CSVs are saved here

# TODO: create the LangSmith client and a WRAPPED OpenAI client (so every call is traced)
client = ...
openai_client = ...

# TODO: verify the connection, e.g. print how many datasets you can see
...

## Part 2: Dataset creation (60–90 min)
### Step 3: Extract and structure your examples

Turn your documents into a list of examples. Every example is a dictionary with the **same keys**.

| Field | Goes to | Example |
|---|---|---|
| `question` (+ `context` if RAG-style) | **inputs**: what the model sees | "Can I return a sale item?" |
| `answer` | **outputs**: the reference answer | "Yes, within 14 days, store credit only." |
| `category`, `difficulty`, `source` | **metadata**: for analysis, the model never sees it | "returns", "hard", "policy.pdf p.3" |

 **Hint:** aim for **15–20 examples** (minimum 10). Include at least 2 categories and at least 2 "hard" or "trap" examples.

In [ ]:
# TODO: build your examples (by hand, or parse a CSV / JSON / text file)
# Option A: write them by hand
examples = [
    {
        "question":   "...",
        "context":    "...",   # optional: delete this key everywhere if you don't use context
        "answer":     "...",
        "category":   "...",
        "difficulty": "...",   # e.g. easy / medium / hard
        "source":     "...",
    },
    # ...
]

# Option B: load from a file
# examples = pd.read_csv("my_examples.csv").to_dict(orient="records")

In [ ]:
# TODO: data quality checks
# 1. At least 10 examples?
# 2. Every example has the same keys, and no empty question / answer?
# 3. How many examples per category and per difficulty?
...

** My data structure:** what each field means, and anything you removed or changed.

### Step 4: Create the LangSmith dataset

 **Hints (§2 of `02_langsmith(1).ipynb`):**
- Check `client.list_datasets()` first, so re-running the cell doesn't create a duplicate.
- `client.create_dataset(name, description=...)` then `client.create_examples(inputs=[...], outputs=[...], metadata=[...], dataset_id=...)`.
- Put a **version** in the name (e.g. `-v1`). If you change the examples later, create `-v2` instead of editing silently.

In [ ]:
DATASET_NAME = ...          # e.g. "support-faq-v1"
DATASET_DESCRIPTION = ...

# TODO: create the dataset only if it doesn't exist yet, then upload inputs, outputs AND metadata
existing = ...
if DATASET_NAME not in existing:
    dataset = ...
    client.create_examples(
        inputs   = ...,     # [{"question": ..., "context": ...}, ...]
        outputs  = ...,     # [{"answer": ...}, ...]
        metadata = ...,     # [{"category": ..., "difficulty": ..., "source": ...}, ...]
        dataset_id = dataset.id,
    )
    print(f"Created '{DATASET_NAME}' with {len(examples)} examples")
else:
    print(f"'{DATASET_NAME}' already exists, skipping upload")

# TODO: verify: count the examples now stored in LangSmith (client.list_examples(dataset_name=...))
...

**Go and look:** open **Datasets → your dataset** in LangSmith. Are inputs, reference outputs and metadata all there? Take a **screenshot** for your repo.

** Dataset name + description:**

## Part 3: Evaluation setup (45–60 min)
### Step 5: Target function

The target is **the system under test**: it takes `inputs` from one example and returns `outputs`.

 **Hints (§3 of `02_langsmith(1).ipynb`):**
- Signature: `def target(inputs: dict) -> dict`, returning e.g. `{"answer": "..."}`.
- Add `@traceable` so the whole function (not only the OpenAI call) appears as one trace.
- Set `temperature=0` for controlled randomness (§5 Reproducibility: still not 100% guaranteed).
- **gpt-5 family:** use `max_completion_tokens` instead of `max_tokens`, and `reasoning_effort="none"` for short direct answers.
- **Error handling:** wrap the API call in `try/except` and return `{"answer": "", "error": str(e)}`, so one failure doesn't stop the whole experiment.

In [ ]:
MODEL = "gpt-4o-mini"

SYSTEM_PROMPT = '''
TODO: instructions for YOUR domain.
What should the model do? What format? What if the answer is not in the context?
'''

@traceable(name="lab-target")
def target(inputs: dict) -> dict:
    # TODO: build the messages from inputs (question, and context if you use it)
    # TODO: call openai_client.chat.completions.create(...) with MODEL and temperature=0
    # TODO: return {"answer": ...}; on an exception return {"answer": "", "error": str(e)}
    ...

In [ ]:
# TODO: test the target on 2-3 examples BEFORE running the full evaluation
for ex in examples[:3]:
    ...

### Step 6: Evaluators

 **Hints:**
- Correctness: OpenEvals `CORRECTNESS_PROMPT` + `create_llm_as_judge(...)`, exactly as in §3 of `02_langsmith(1).ipynb`.
- Use a **different (stronger) model as judge** than the model under test.
- Exact match only makes sense when answers are short labels (a letter, a category, a number). For free-text answers, skip it or use a keyword check instead.
- An evaluator function returns a dict like `{"key": "my_metric", "score": True}`.

In [ ]:
JUDGE_MODEL = ...           # e.g. "openai:gpt-5.4-mini"

# TODO: correctness evaluator (LLM-as-a-judge)
_correctness_judge = ...

def correctness_evaluator(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    ...

# TODO (pick ONE that fits your domain): a cheap, deterministic evaluator without an LLM
# e.g. exact match for labels, or "does the answer contain the key term(s)?"
def rule_evaluator(outputs: dict, reference_outputs: dict) -> dict:
    ...

In [ ]:
# TODO: test your evaluators on 1 correct and 1 deliberately wrong answer
...

** What does "correct" mean in my domain?** One correct answer or many valid ones? What does each evaluator measure?

## Part 4: Run the evaluation (30–45 min)
### Step 7: Execute the experiment

 **Hint (§3 of `02_langsmith(1).ipynb`):** `client.evaluate(target, data=..., evaluators=[...], experiment_prefix=..., max_concurrency=2)`. Use a prefix that tells you **what** ran, e.g. `"gpt-4o-mini-promptv1"`.

In [ ]:
# TODO: run the experiment
results = ...
print(results)

In [ ]:
# TODO: collect the results into a DataFrame and check that EVERY example was processed
df = results.to_pandas()
print(df.columns.tolist())    # find the inputs.*, outputs.*, reference.*, feedback.* columns
...
df.to_csv(OUT / "results_main.csv", index=False)

### Step 8: Review the results in the LangSmith UI

**Go and look:** click the experiment link printed above.
- Open 2–3 **failed** examples and read the judge's reasoning (`comment`).
- Do the LLM judge and your rule-based evaluator **agree**? Look at the rows where they don't.
- Take **screenshots** (experiment table + one trace) for your repo.

** First observations + experiment link:**

## Part 5: Analysis & reporting (45–60 min)
### Step 9: Analyse results

In [ ]:
# TODO: aggregate metrics for each evaluator: mean (= pass rate for True/False), n, number failed
#  True/False scores: .astype(float).mean() gives the pass rate
...

In [ ]:
# TODO: performance by category and by difficulty
# The results DataFrame has no metadata, so join it from the dataset:
meta = pd.DataFrame([
    {"example_id": str(e.id), **(e.metadata or {})}
    for e in client.list_examples(dataset_name=DATASET_NAME)
])
df["example_id"] = df["example_id"].astype(str)
df = df.merge(meta, on="example_id", how="left")

# TODO: groupby category / difficulty -> pass rate; bar chart saved to OUT / "pass_rate_by_category.png"
...

In [ ]:
# TODO: error analysis
# 1. Show every failed example: question, model answer, reference, judge comment
# 2. Give each failure a type by hand: wrong fact / incomplete / hallucination / wrong format / judge mistake / bad reference
...

** Key findings:**
1. Overall pass rate per evaluator:
2. Easiest and hardest category:
3. Biggest failure pattern (with one example):
4. Did any failure turn out to be a **judge mistake** or a **bad reference answer**?

 **Remember (W7/D5 statistics):** with 15–20 examples, every score is an **estimate**. One extra failure moves the pass rate by 5–7 points.

### Step 10: `evaluation_summary.md`

**One short paragraph** (plus optional bullets/table): what you evaluated, main metrics, biggest failure pattern, limitations, one recommendation. Not a long report.

In [ ]:
# TODO: fill in and save. It goes at the ROOT of your repo.
summary = '''# Evaluation summary

TODO: one paragraph: domain + dataset (size, categories), model + settings, evaluators,
main metrics, biggest failure pattern, limitations, one recommendation.

| Metric | Score |
|---|---|
| Correctness (LLM judge) | ... |
| ... | ... |

LangSmith experiment: <link>
'''
Path("evaluation_summary.md").write_text(summary, encoding="utf-8")

---
## Optional Part 1: Custom evaluator (60–90 min)
### Step 11: Design it

Pick **one** dimension beyond correctness that matters for your domain: tone, completeness, groundedness in the context, safety / policy compliance, format…

 **Hints:**
- Use the 4-part judge prompt anatomy from notebook 01, §3: task, what you're given, criteria, reasoning steps + output format.
- Your prompt's `{variables}` must match the keyword arguments you pass (`inputs`, `outputs`, `reference_outputs`, or your own like `context`).
- Prebuilt prompts you can adapt: `RAG_GROUNDEDNESS_PROMPT`, `ANSWER_RELEVANCE_PROMPT`, `CONCISENESS_PROMPT`, `TOXICITY_PROMPT` (`print()` one to see how it's written).
- For a 1–5 scale instead of True/False: `create_llm_as_judge(..., continuous=True)` gives a 0–1 score, or `choices=[1, 2, 3, 4, 5]`.

In [ ]:
CUSTOM_PROMPT = '''
TODO: your judge prompt with {inputs}, {outputs} (and {reference_outputs} if needed)
'''

custom_judge = ...

def custom_evaluator(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    ...

# TODO: test on 2-3 hand-made cases (one good, one bad, one borderline)
...

### Step 12: Run it next to correctness

In [ ]:
# TODO: new experiment with correctness + custom evaluator, then compare the two scores per example
results_custom = ...
...
# Which examples are CORRECT but score badly on your custom dimension (or the other way round)?
...

### Step 13:  Document your custom evaluator
- What it measures and why it matters in my domain:
- Scoring scale and criteria:
- Where it disagreed with correctness, and what that revealed:

---
## Optional Part 2: Cost & performance (60–90 min)
### Step 14: A/B comparison

 **Hints (§4 of `02_langsmith(1).ipynb`):**
- Same dataset, same evaluators, **only one thing changes** (the model, or the temperature, or the prompt).
- To get token usage, also return it from the target: `{"answer": ..., "prompt_tokens": r.usage.prompt_tokens, "completion_tokens": r.usage.completion_tokens}`.
- Latency: look for an `execution_time` column in `results.to_pandas()`.
- Fill `PRICES` from the **OpenAI pricing page** (USD per 1M tokens). Write down the date you checked it.

In [ ]:
CONFIGS = {
    # experiment name: model
    "config-a": "gpt-4o-mini",
    "config-b": ...,
    # "config-c": ...,
}

PRICES = {   # USD per 1M tokens, from the OpenAI pricing page (checked on: ...)
    "gpt-4o-mini": {"input": ..., "output": ...},
    # ...
}

def make_target(model):
    @traceable(name=f"lab-target-{model}")
    def _target(inputs: dict) -> dict:
        # TODO: same as your target, but with `model` and token usage in the returned dict
        ...
    return _target

# TODO: run one experiment per config and keep the results
ab_results = {}
...

### Step 15: Cost vs performance

In [ ]:
# TODO: one row per config: pass rate, avg latency, total cost, cost per example, cost per CORRECT answer
# cost = prompt_tokens / 1e6 * price_in + completion_tokens / 1e6 * price_out
...
# TODO: scatter plot: x = cost per example, y = pass rate, one labelled point per config
...
plt.savefig(OUT / "cost_vs_performance.png", dpi=120)
plt.show()

 **Before you recommend, think:**
- Is the gap between models bigger than **one or two examples**? With ~20 examples it may be luck (W7/D5).
- Which pass rate is **good enough** for this use case? A cheaper model that meets that bar wins.

### Step 16: `optimization_summary.md`

In [ ]:
# TODO: one short paragraph: best cost/performance trade-off, and when you would pick each config
opt = '''# Optimization summary

TODO: best trade-off + when to pick each configuration.

| Config | Model | Pass rate | Cost / correct answer | Avg latency |
|---|---|---|---|---|
| ... | ... | ... | ... | ... |
'''
Path("optimization_summary.md").write_text(opt, encoding="utf-8")

---
## Reflection (optional, recommended)
1. What surprised you when you read the failed traces?
2. Did you trust your LLM judge? How did you check it?
3. What would you change in your dataset for a `-v2`?

## Submission checklist
- [ ] GitHub repo with **only** this lab, and a `README.md` (domain + dataset, approach, how to run, LangSmith links, file map)
- [ ] `lab_proof.md` as the **entry point**: code/workflow, one run with its input, the evidence it produced (trace / screenshot), the final output, and one failure or limitation you hit
- [ ] `evaluation_summary.md` at the repo root (one paragraph + optional table)
- [ ] (Optional) `optimization_summary.md` at the repo root
- [ ] Notebook runs top to bottom without errors; descriptive file names (no `lab.ipynb`, `final_v2.py`)
- [ ] `outputs/`: results CSV(s), charts, and LangSmith screenshots (dataset, experiment, one trace)
- [ ] **No API keys** in the repo: `.env` is in `.gitignore`